In [1]:
import pandas as pd
import sys, os
sys.path.append(os.path.abspath("../data_creation/plots_and_analysis"))
import config
from pathlib import Path
import numpy as np
# adjust paths in /data_creation/plots_and_analysis/config.py

In [14]:
config.path_to_evaluation

'/home/dev/projects/data/evaluation/'

In [3]:
runs = ["case1_run1", "case1_run2", "case1_run3", "case1_run4", "case1_run5"]

In [4]:
metrics = ["NRMSE", "MASE", "sMAPE"]
p_list = ["5000", "12000", "25000", "50000", "100000", "all"]
#["100", "200", "350", "all"] 

Plot error distributions across potential target sizes for all runs for each div step

In [ ]:
# Create a table with RMSSE, MASE, sMAPE, potential target size, and similarity for each div step
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

for run in runs:
    path_to_forecasts = f"{config.path_to_data}/{run}/outputs/forecasts/"

    for div_step in [10, 5, 0]:
        # Create evaluation directory (for tables and plots)
        eval_dir = Path(path_to_forecasts) / "evaluation"
        eval_dir.mkdir(parents=True, exist_ok=True)

        # ------------------------------------------------
        # TABLE SECTION
        # ------------------------------------------------
        div_step_mets = []

        for p in p_list:
            path = Path(path_to_forecasts) / f"target_div{div_step}_sim{p}{run}"
            file_path = next(path.glob("*item_metrics.parquet"))

            single_mets = pd.read_parquet(file_path)[metrics]
            single_mets = (
                single_mets
                .replace([np.inf, -np.inf], np.nan)  # Replace infinite values with NaN
                .dropna(subset=metrics)             # Remove rows containing NaN values in metrics
            )

            single_mets_median = single_mets.median().to_frame().T
            single_mets_median["pot target size"] = p
            div_step_mets.append(single_mets_median)

        # Load source metrics
        source_path = Path(path_to_forecasts) / f"source_div{div_step}{run}"
        source_file_path = next(source_path.glob("*item_metrics.parquet"))

        source_mets = pd.read_parquet(source_file_path)[metrics]
        source_mets = (
            source_mets
            .replace([np.inf, -np.inf], np.nan)  # Replace infinite values with NaN
            .dropna(subset=metrics)             # Remove rows containing NaN values in metrics
        )

        source_mets_median = source_mets.median().to_frame().T
        source_mets_median["pot target size"] = "source"
        div_step_mets.append(source_mets_median)

        mets = pd.concat(div_step_mets)[["NRMSE", "MASE", "sMAPE", "pot target size"]]
        mets.rename(columns={"NRMSE": "RMSSE"}, inplace=True)

        cols_to_round = ["RMSSE", "MASE", "sMAPE"]
        mets[cols_to_round] = mets[cols_to_round].round(3)

        # Merge similarity information
        sim_path = Path(config.path_to_data) / f"{run}/interim/source_div{div_step}{run}/similarity.parquet"
        similarity = pd.read_parquet(sim_path)
        similarity["similarity"] = similarity["similarity"].round(2)

        mets = mets.merge(similarity, on="pot target size")

        mets.to_parquet(eval_dir / f"metrics_div_step{div_step}.parquet")
        mets.to_csv(eval_dir / f"metrics_div_step{div_step}.csv", index=False)

        # ------------------------------------------------
        # BOXPLOT SECTION
        # ------------------------------------------------
        # For the boxplots, we need the full metric distributions:
        # Dictionary structure: metric -> list of arrays (one per target size + source)
        data_for_plot = {m: [] for m in metrics}
        labels = []

        # Load target datasets
        for p in p_list:
            path = Path(path_to_forecasts) / f"target_div{div_step}_sim{p}{run}"
            file_path = next(path.glob("*item_metrics.parquet"))

            df = pd.read_parquet(file_path)[metrics]
            df = (
                df.replace([np.inf, -np.inf], np.nan)  # Replace infinite values with NaN
                  .dropna(subset=metrics)             # Remove rows containing NaN values in metrics
            )

            for m in metrics:
                data_for_plot[m].append(df[m].values)
                print(f"div {div_step}, p {p}, metric {m}: {df[m].values}")  # Debug output

            labels.append(p)

        # Load source dataset
        df_source = pd.read_parquet(source_file_path)[metrics]
        df_source = (
            df_source
            .replace([np.inf, -np.inf], np.nan)  # Replace infinite values with NaN
            .dropna(subset=metrics)             # Remove rows containing NaN values in metrics
        )

        for m in metrics:
            data_for_plot[m].append(df_source[m].values)

        labels.append("source")

        # Create three side-by-side boxplots (RMSSE / MASE / sMAPE)
        fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharex=True)

        for ax, m in zip(axes, metrics):
            ax.boxplot(data_for_plot[m], labels=labels)

            if m == "NRMSE":
                ax.set_ylabel("RMSSE", fontsize=16, fontweight="bold")
            else:
                ax.set_ylabel(m, fontsize=16, fontweight="bold")

            ax.set_xlabel("Potential target size / source", fontsize=16, fontweight="bold")
            ax.tick_params(axis="x", rotation=45, labelsize=14)

            # Use a fixed y-axis range for easier comparison
            if m in ["NRMSE", "MASE"]:
                ax.set_ylim(0, 5)

        fig.tight_layout()
        fig.savefig(eval_dir / f"boxplot_{run}_div{div_step}.pdf", dpi=300)
        plt.close(fig)

div 10, p 5000, metric NRMSE: [0.64229668 0.06767992 4.04668594 ... 0.81411935 0.91187587 0.43329621]
div 10, p 5000, metric MASE: [1.29508788 0.19688325 4.22421524 ... 0.70017274 1.0463004  0.52756784]
div 10, p 5000, metric sMAPE: [0.29354016 0.27134927 0.56451638 ... 0.17742021 0.25135503 0.10440047]
div 10, p 12000, metric NRMSE: [0.20075027 0.17218002 0.56672822 ... 0.22538075 8.74828549 0.78696031]
div 10, p 12000, metric MASE: [ 0.32110316  0.25979591  0.95311255 ...  0.42969178 10.86452508
  0.77638709]
div 10, p 12000, metric sMAPE: [0.1368078  0.36020493 0.35543371 ... 0.23053376 0.96788048 0.16007408]
div 10, p 25000, metric NRMSE: [0.31484728 2.05329241 1.15697172 ... 0.20242931 0.78696031 0.91187587]
div 10, p 25000, metric MASE: [1.1029952  1.98881243 1.18815976 ... 0.66518869 0.77638709 1.0463004 ]
div 10, p 25000, metric sMAPE: [0.40261493 0.684021   0.22368416 ... 0.2340596  0.16007408 0.25135503]
div 10, p 50000, metric NRMSE: [0.47704039 1.29305344 0.23884918 ... 1.5

/tmp/ipykernel_348596/3830847879.py:107: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data_for_plot[m], labels=labels)
/tmp/ipykernel_348596/3830847879.py:107: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data_for_plot[m], labels=labels)
/tmp/ipykernel_348596/3830847879.py:107: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data_for_plot[m], labels=labels)


div 5, p 5000, metric NRMSE: [0.70259582 0.14070183 1.29659685 ... 1.02381745 0.83958956 0.53553917]
div 5, p 5000, metric MASE: [1.14905741 0.31388657 1.49723128 ... 1.28841242 0.87786504 0.6113604 ]
div 5, p 5000, metric sMAPE: [0.38669816 0.11147599 0.20212482 ... 0.14260049 0.20538725 0.12122138]
div 5, p 12000, metric NRMSE: [0.58260424 0.72013696 0.20786414 ... 8.77635934 0.02893248 0.83958956]
div 5, p 12000, metric MASE: [ 0.67513858  0.90070367  0.46032114 ... 10.88567175  0.08884605
  0.87786504]
div 5, p 12000, metric sMAPE: [0.63114584 0.05109363 0.52564718 ... 0.96756636 0.2157079  0.20538725]
div 5, p 25000, metric NRMSE: [0.05798401 0.2789845  0.93345571 ... 0.17410366 0.16688579 0.91941203]
div 5, p 25000, metric MASE: [0.18665475 0.57863901 1.13294952 ... 0.36833134 0.53889387 0.98669885]
div 5, p 25000, metric sMAPE: [0.56728259 0.27154665 0.2020698  ... 0.29562671 0.19359613 0.36846769]
div 5, p 50000, metric NRMSE: [0.34997544 0.7111128  1.76741475 ... 0.78314174 0.

/tmp/ipykernel_348596/3830847879.py:107: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data_for_plot[m], labels=labels)
/tmp/ipykernel_348596/3830847879.py:107: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data_for_plot[m], labels=labels)
/tmp/ipykernel_348596/3830847879.py:107: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data_for_plot[m], labels=labels)


div 0, p 5000, metric NRMSE: [0.78279831 0.14305144 0.29119325 ... 0.79564993 0.94173605 0.91241431]
div 0, p 5000, metric MASE: [1.25283322 0.45731716 0.3276647  ... 0.71322451 1.23044025 1.02363795]
div 0, p 5000, metric sMAPE: [0.41312568 1.7124841  0.82579655 ... 0.18147129 0.13551841 0.24554899]
div 0, p 12000, metric NRMSE: [0.1244502  0.02314259 3.42907451 ... 0.24168802 0.02938382 0.79564993]
div 0, p 12000, metric MASE: [0.29142253 0.07753876 3.68411127 ... 0.46946917 0.08758357 0.71322451]
div 0, p 12000, metric sMAPE: [0.64770032 0.41711457 0.44826138 ... 0.2499967  0.21267574 0.18147129]
div 0, p 25000, metric NRMSE: [0.09944071 1.10829286 0.00347758 ... 1.08203087 0.24885592 0.14618569]
div 0, p 25000, metric MASE: [0.3166739  1.05250968 0.01697937 ... 1.26173112 0.6765433  0.26313025]
div 0, p 25000, metric sMAPE: [0.31114151 0.16851324 0.29317093 ... 0.1022667  1.25377854 0.18707025]
div 0, p 50000, metric NRMSE: [1.06274561 1.27869098 0.00132028 ... 0.2150931  0.0698877

...

/tmp/ipykernel_348596/3830847879.py:107: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data_for_plot[m], labels=labels)
/tmp/ipykernel_348596/3830847879.py:107: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data_for_plot[m], labels=labels)
/tmp/ipykernel_348596/3830847879.py:107: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data_for_plot[m], labels=labels)


div 5, p 5000, metric NRMSE: [0.0646104  0.70882098 0.02398507 ... 0.79672429 0.91025094 0.43481638]
div 5, p 5000, metric MASE: [0.1633747  0.70274251 0.0681065  ... 0.69270527 1.06062762 0.56967388]
div 5, p 5000, metric sMAPE: [0.16235215 0.11928256 0.84769824 ... 0.17603888 0.25623435 0.11281824]
div 5, p 12000, metric NRMSE: [0.01410015 0.1216172  0.00887564 ... 0.74013086 0.79672429 0.43481638]
div 5, p 12000, metric MASE: [0.05504381 0.18902028 0.02953608 ... 0.78625052 0.69270527 0.56967388]
div 5, p 12000, metric sMAPE: [0.33736741 0.22814646 0.27433003 ... 0.16222667 0.17603888 0.11281824]
div 5, p 25000, metric NRMSE: [1.32186099 1.51442779 1.64016843 ... 0.06977017 0.92035891 0.74013086]
div 5, p 25000, metric MASE: [1.66072236 1.55149944 1.96521685 ... 0.22054453 0.96545959 0.78625052]
div 5, p 25000, metric sMAPE: [0.28064505 0.29274775 0.48126525 ... 0.22486869 0.34947258 0.16222667]
div 5, p 50000, metric NRMSE: [0.21649533 0.37728665 0.06537883 ... 2.41715547 0.5571996

/tmp/ipykernel_348596/3830847879.py:107: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data_for_plot[m], labels=labels)
/tmp/ipykernel_348596/3830847879.py:107: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data_for_plot[m], labels=labels)
/tmp/ipykernel_348596/3830847879.py:107: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data_for_plot[m], labels=labels)


div 0, p 5000, metric NRMSE: [0.25542604 1.52306177 0.31145927 ... 1.04925677 0.91201055 0.55265009]
div 0, p 5000, metric MASE: [0.54604916 1.65830492 0.54176207 ... 1.30209518 0.95949799 0.66287567]
div 0, p 5000, metric sMAPE: [0.4173483  0.17617208 0.29883428 ... 0.14390918 0.22688161 0.13243751]
div 0, p 12000, metric NRMSE: [0.00875856 0.85802292 0.11564553 ... 0.77282345 1.04925677 0.91201055]
div 0, p 12000, metric MASE: [0.02828244 1.42613138 0.3674502  ... 0.79987147 1.30209518 0.95949799]
div 0, p 12000, metric sMAPE: [0.26480712 0.48051065 0.98751589 ... 0.16426312 0.14390918 0.22688161]
div 0, p 25000, metric NRMSE: [2.07795685 1.98600888 0.52099895 ... 0.07097723 0.63641154 0.21063155]
div 0, p 25000, metric MASE: [3.81934379 1.84812105 0.71854276 ... 0.21705293 0.73199532 0.46435562]
div 0, p 25000, metric sMAPE: [0.91297649 0.23854303 0.496911   ... 0.2194962  0.08458868 0.2472928 ]
div 0, p 50000, metric NRMSE: [0.32854012 0.67027676 0.21357309 ... 0.03501231 0.9214530

/tmp/ipykernel_348596/3830847879.py:107: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data_for_plot[m], labels=labels)
/tmp/ipykernel_348596/3830847879.py:107: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data_for_plot[m], labels=labels)
/tmp/ipykernel_348596/3830847879.py:107: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data_for_plot[m], labels=labels)


Save single metrics across OSDs and pot target sizes per run

In [ ]:
all_mets = []
for run in runs:
    path_to_forecasts = f"{config.path_to_data}/{run}/outputs/forecasts/"

    all_div_step_mets = []

    for div_step in [10, 5, 0]:
        # Create evaluation directory (for tables and plots)
        eval_dir = Path(path_to_forecasts) / "evaluation"
        eval_dir.mkdir(parents=True, exist_ok=True)

        # ------------------------------------------------
        # TABLE SECTION
        # ------------------------------------------------
        div_step_mets = []

        # Load metrics for all potential target sizes
        for p in p_list:
            path = Path(path_to_forecasts) / f"target_div{div_step}_sim{p}{run}"
            file_path = next(path.glob("*item_metrics.parquet"))

            single_mets = pd.read_parquet(file_path)[metrics + ["item_id"]]
            single_mets = (
                single_mets
                .replace([np.inf, -np.inf], np.nan)  # Replace infinite values with NaN
                .dropna(subset=metrics)             # Remove rows containing NaN values in metrics
            )

            single_mets["pot target size"] = p
            single_mets["run"] = run
            div_step_mets.append(single_mets)

        # Load source metrics
        source_path = Path(path_to_forecasts) / f"source_div{div_step}{run}"
        source_file_path = next(source_path.glob("*item_metrics.parquet"))

        source_mets = pd.read_parquet(source_file_path)[metrics + ["item_id"]]
        source_mets = (
            source_mets
            .replace([np.inf, -np.inf], np.nan)  # Replace infinite values with NaN
            .dropna(subset=metrics)             # Remove rows containing NaN values in metrics
        )

        source_mets["pot target size"] = "source"
        source_mets["run"] = run
        div_step_mets.append(source_mets)

        # Combine all metrics for the current div step
        mets = pd.concat(div_step_mets)[["NRMSE", "MASE", "sMAPE", "pot target size", "item_id", "run"]]
        mets.rename(columns={"NRMSE": "RMSSE"}, inplace=True)

        cols_to_round = ["RMSSE", "MASE", "sMAPE"]
        mets[cols_to_round] = mets[cols_to_round].round(3)

        # Add div step as an additional column
        mets["div_step"] = div_step

        all_div_step_mets.append(mets)

    # Combine metrics from all div steps
    mets_for_stats = pd.concat(all_div_step_mets)
    all_mets.append(mets_for_stats)

    # Save metrics for statistical analysis
    #mets_for_stats.to_csv(eval_dir / f"single_mets_{run}.csv", index=False)
all_mets_df = pd.concat(all_mets)
all_mets_df.to_parquet(f"{config.path_to_evaluation}/all_single_metrics.parquet", index=False)


In [16]:
all_mets_df

,RMSSE,MASE,sMAPE,pot target size,item_id,run,div_step
0,0.642,1.295,0.294,5000,T94287_kaggle,case1_run1,10
1,0.068,0.197,0.271,5000,T94288_kaggle,case1_run1,10
2,4.047,4.224,0.565,5000,T94289_kaggle,case1_run1,10
3,0.525,0.928,0.251,5000,T9428_kaggle,case1_run1,10
4,1.353,1.974,0.172,5000,T94290_kaggle,case1_run1,10
...,...,...,...,...,...,...,...
4995,0.653,1.143,0.260,source,T94242_kaggle,case1_run5,0
4996,1.759,1.894,0.374,source,T94256_kaggle,case1_run5,0
4997,0.126,0.397,0.950,source,T94282_kaggle,case1_run5,0
4998,0.846,0.800,0.080,source,T94317_kaggle,case1_run5,0


In [17]:
all_mets_df[['run', 'div_step']].value_counts()

run         div_step
case1_run2  0           34912
case1_run3  5           34909
            10          34909
case1_run1  10          34908
case1_run4  5           34908
case1_run1  5           34906
case1_run2  5           34901
case1_run4  10          34901
case1_run5  5           34900
case1_run1  0           34897
case1_run4  0           34897
case1_run5  10          34897
            0           34896
case1_run3  0           34892
case1_run2  10          34890
dtype: int64